In [176]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import Dropout
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split

In [ ]:
df=pd.read_csv("FA-KES-Dataset.csv",encoding="latin-1")
df.head()

,unit_id,article_title,article_content,source,date,location,labels
0,1914947530,Syria attack symptoms consistent with nerve ag...,Wed 05 Apr 2017 Syria attack symptoms consiste...,nna,4/5/2017,idlib,0
1,1914947532,Homs governor says U.S. attack caused deaths b...,Fri 07 Apr 2017 at 0914 Homs governor says U.S...,nna,4/7/2017,homs,0
2,1914947533,Death toll from Aleppo bomb attack at least 112,Sun 16 Apr 2017 Death toll from Aleppo bomb at...,nna,4/16/2017,aleppo,0
3,1914947534,Aleppo bomb blast kills six Syrian state TV,Wed 19 Apr 2017 Aleppo bomb blast kills six Sy...,nna,4/19/2017,aleppo,0
4,1914947535,29 Syria Rebels Dead in Fighting for Key Alepp...,Sun 10 Jul 2016 29 Syria Rebels Dead in Fighti...,nna,7/10/2016,aleppo,0


In [178]:
df=df.dropna()

In [179]:
X=df.drop('labels',axis=1)

In [180]:
y=df['labels']

In [181]:
X.shape,y.shape

((804, 6), (804,))

In [182]:
voc_size=5000

In [183]:
messages=X.copy()

In [184]:
messages['article_title'][1]

'Homs governor says U.S. attack caused deaths but doesnt see big human losses'

In [185]:
messages.reset_index(inplace=True)

In [186]:
nltk.download('stopwords')

[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


False

In [187]:
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['article_title'][i])
    review = review.lower()
    review = review.split()
    
    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [188]:
corpus

['syria attack symptom consist nerv agent use',
 'hom governor say u attack caus death doesnt see big human loss',
 'death toll aleppo bomb attack least',
 'aleppo bomb blast kill six syrian state tv',
 'syria rebel dead fight key aleppo road',
 'suicid bomb kill least northeast syria',
 'dead heavi u raid syria stronghold',
 'suicid bomber kill assad clan hometown',
 'explos rock town damascu',
 'damascu explos due rocket bomb',
 'syrian regim step aerial assault douma',
 'hizballah lead regim offens southern syria',
 'syrian opposit remain divid',
 'video show murder syrian activist',
 'syria nusra front stage deadli suicid bomb aleppo',
 'regim troop thwart rebel attack syria aleppo',
 'ahrar al sham leader kill syria',
 'barrel bomb kill town syria',
 'rebel advanc north western syria',
 'isra strike syrian town kill pro regim fighter',
 'syria armi plane crash rebel held town',
 'syrian regim reveng attack kill score qalamoun',
 'chemic massacr idlib defi world',
 'bu bomb mark en

In [189]:
onehot_repr=[one_hot(words,voc_size)for words in corpus] 
onehot_repr

[[1849, 2326, 1153, 2827, 4359, 4943, 2469],
 [240, 4067, 2572, 862, 2326, 896, 4188, 3330, 267, 2671, 1285, 1534],
 [4188, 1559, 2805, 1670, 2326, 1801],
 [2805, 1670, 735, 4454, 1566, 4240, 3976, 3425],
 [1849, 1231, 2840, 2482, 883, 2805, 448],
 [1401, 1670, 4454, 1801, 117, 1849],
 [2840, 2941, 862, 1236, 1849, 4724],
 [1401, 1277, 4454, 2070, 3025, 1405],
 [4224, 2719, 3738, 2261],
 [2261, 4224, 426, 2215, 1670],
 [4240, 3194, 4112, 2166, 1144, 4317],
 [1283, 3247, 3194, 3082, 2745, 1849],
 [4240, 2539, 3333, 2923],
 [3166, 4970, 563, 4240, 1165],
 [1849, 837, 1890, 4475, 801, 1401, 1670, 2805],
 [3194, 217, 3908, 1231, 2326, 1849, 2805],
 [1812, 1302, 1160, 4195, 4454, 1849],
 [4601, 1670, 4454, 3738, 1849],
 [1231, 3506, 147, 3926, 1849],
 [4492, 4826, 4240, 3738, 4454, 244, 3194, 4703],
 [1849, 381, 4758, 3800, 1231, 1618, 3738],
 [4240, 3194, 213, 2326, 4454, 295, 4317],
 [4989, 1141, 3527, 2238, 1391],
 [836, 1670, 857, 4005, 4775, 2031, 4667, 1849],
 [1566, 4454, 2805, 1670,

In [190]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ... 4359 4943 2469]
 [   0    0    0 ... 2671 1285 1534]
 [   0    0    0 ... 1670 2326 1801]
 ...
 [   0    0    0 ... 4240 2805 1440]
 [   0    0    0 ... 3913 4956 1849]
 [   0    0    0 ... 4737 3296 2803]]


In [191]:
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0, 1849, 2326, 1153, 2827, 4359, 4943, 2469], dtype=int32)

In [192]:
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Bidirectional(LSTM(100)))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [193]:
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [194]:
X_final.shape,y_final.shape

((804, 20), (804,))

In [195]:
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

In [196]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=16)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.4966 - loss: 0.6942 - val_accuracy: 0.5564 - val_loss: 0.6904
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5406 - loss: 0.6914 - val_accuracy: 0.5113 - val_loss: 0.6925
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6081 - loss: 0.6804 - val_accuracy: 0.4699 - val_loss: 0.6988
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7426 - loss: 0.6265 - val_accuracy: 0.5038 - val_loss: 0.7733
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7610 - loss: 0.5322 - val_accuracy: 0.5000 - val_loss: 0.7894
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8029 - loss: 0.4427 - val_accuracy: 0.4925 - val_loss: 0.8842
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8339 - loss: 0.3714 - val_accuracy: 0.4962 - val_loss: 1.0767
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8627 - loss: 0.3008 - val_accuracy: 0.4850 - v

In [197]:
y_pred_proba=model.predict(X_test)
y_pred_proba

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step


array([[7.71096051e-01],
       [9.94659722e-01],
       [9.96064186e-01],
       [6.14096299e-02],
       [9.21339750e-01],
       [8.17641854e-01],
       [9.88349199e-01],
       [9.34799135e-01],
       [8.84976506e-01],
       [9.76507664e-01],
       [5.32611981e-02],
       [9.52177703e-01],
       [7.59706557e-01],
       [9.58016142e-02],
       [3.34115446e-01],
       [9.90855455e-01],
       [9.28152442e-01],
       [3.65873963e-01],
       [6.25182450e-01],
       [6.26594961e-01],
       [9.93931592e-01],
       [8.97038758e-01],
       [7.16764450e-01],
       [9.11867321e-01],
       [6.71168149e-01],
       [4.27278429e-02],
       [8.26263487e-01],
       [7.99845040e-01],
       [8.55790198e-01],
       [8.89395848e-02],
       [9.78560090e-01],
       [9.34818387e-01],
       [5.67445517e-01],
       [9.56202865e-01],
       [2.62434602e-01],
       [2.38373280e-01],
       [9.96967077e-01],
       [4.57577974e-01],
       [9.28156614e-01],
       [9.90487695e-01],


In [198]:
y_pred_class=np.argmax(y_pred_proba,axis=1)
y_pred_class

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0])

In [199]:
confusion_matrix(y_test,y_pred_class)

array([[118,   0],
       [148,   0]])

In [200]:
accuracy_score(y_test,y_pred_class)

0.44360902255639095

In [201]:
print(classification_report(y_test,y_pred_class))

              precision    recall  f1-score   support

           0       0.44      1.00      0.61       118
           1       0.00      0.00      0.00       148

    accuracy                           0.44       266
   macro avg       0.22      0.50      0.31       266
weighted avg       0.20      0.44      0.27       266



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
